In [1]:
pip install 'accelerate>=1.1.0'

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import json , ast
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import Dataset
import torch
from transformers import AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from sklearn.utils.class_weight import compute_class_weight
from transformers import DataCollatorForSeq2Seq
import torch.nn as nn  
from sklearn.metrics import f1_score,accuracy_score,classification_report

In [7]:
SEED = 42

In [8]:
train = pd.read_excel(r'DeepX_train.xlsx')

unlabeled = pd.read_excel(r'DeepX_unlabeled.xlsx')

validation = pd.read_excel(r'DeepX_validation.xlsx')

In [9]:
train.head()

,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]","{""app_experience"": ""negative"", ""delivery"": ""ne..."
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]","{""cleanliness"": ""positive"", ""ambiance"": ""posit..."
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,1,قبل شهر,بيت لحم Beet Lahm,مطعم,google_maps,"[""service"", ""delivery"", ""food""]","{""service"": ""negative"", ""delivery"": ""negative""..."
3,3024,احلي مكان فزايد,5,قبل شهر,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,"[""general""]","{""general"": ""positive""}"
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,4,قبل سنة,The Best Restaurant,مطعم,google_maps,"[""food"", ""price""]","{""food"": ""positive"", ""price"": ""positive""}"


In [10]:
validation.head()

,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,4446,مريم سوتلي الاظافررر تحفههه اوييي ❤️❤️❤️❤️❤️,5,قبل شهرين,Sand salon,صالون تجميل,google_maps,"[""service""]","{""service"": ""positive""}"
1,8612,التطبيق جميل .. أتمنى إضافة البحث عن طريق الخر...,4,2020-10-28 00:00:00,Aqarmap,real_estate,play_store,"[""app_experience""]","{""app_experience"": ""neutral""}"
2,6729,سراقين مكتوب وصلت السياره والسواق مارضى يقول و...,1,2026-02-04 00:00:00,Careem,transport,play_store,"[""service"", ""delivery"", ""price""]","{""service"": ""negative"", ""delivery"": ""negative""..."
3,6292,سي جيدا,1,2025-08-07 00:00:00,Elmenus,food_delivery,play_store,"[""general""]","{""general"": ""negative""}"
4,1639,مكان ممتاز جدا و الخدمة جيده جدا,4,قبل أسبوع,Holm Cafe,مقهى,google_maps,"[""ambiance"", ""service""]","{""ambiance"": ""positive"", ""service"": ""positive""}"


In [11]:
unlabeled.head()

,review_id,review_text,star_rating,date,business_name,business_category,platform
0,1,Incroyablement grand avec des belles boutiques...,5,قبل 7 ساعات,مول سيتي ستارز.,مركز تسوق,google_maps
1,2,زحمه جدا,5,قبل 12 ساعة,مول سيتي ستارز.,مركز تسوق,google_maps
2,3,حلو فخم كشخة محترم ورايق ينفع للعوائل الخليجي...,5,قبل يوم واحد,مول سيتي ستارز.,مركز تسوق,google_maps
3,4,طبعا غني عن التعريف بتاع البشوات,5,قبل يوم واحد,مول سيتي ستارز.,مركز تسوق,google_maps
4,5,Centro commerciale al Cairo... Molto grande e ...,5,قبل يومين (2),مول سيتي ستارز.,مركز تسوق,google_maps


# Constants 

In [12]:
ASPECTS = [
    "food",
    "service",
    "price",
    "cleanliness",
    "delivery",
    "ambiance",
    "app_experience",
    "general"
]

SENTIMENTS = ["negative", "neutral", "positive"]

sentiment2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2sentiment = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

aspect_ar = {
    "food": "الطعام",
    "service": "الخدمة",
    "price": "السعر",
    "cleanliness": "النظافة",
    "delivery": "التوصيل",
    "ambiance": "المكان والجو العام",
    "app_experience": "تجربة التطبيق",
    "general": "التقييم العام"
}

# Clean the dataset

In [13]:
def clean_text(text):
    text = str(text)

    text = text.replace("ـ", "")
    text = re.sub(r"[\u064B-\u065F]", "", text)

    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)

    text = re.sub(r"http\S+|www\S+", " رابط ", text)
    text = re.sub(r"@\w+", " مستخدم ", text)

    # Keeps Arabic, English, numbers
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

# Parsing and Nulls Handling 

In [14]:
def parse_obj(x):
    if isinstance(x, (list, dict)):
        return x

    if pd.isna(x):
        return None

    x = str(x).strip()

    try:
        return json.loads(x)
    except:
        pass

    try:
        return ast.literal_eval(x)
    except:
        pass

    if "," in x:
        return [item.strip() for item in x.split(",")]

    return x

# Remove the Exact duplicates in the data 

In [15]:
train_raw = train.drop_duplicates(
    subset=["review_text", "aspects", "aspect_sentiments"]
).reset_index(drop=True)


# Explode Labeled data into aspects level raws

In [16]:
def explode_train(df):
    rows = []

    for _, row in df.iterrows():
        review_id = row["review_id"]
        review_text = str(row["review_text"])

        aspects = parse_obj(row["aspects"])
        aspect_sentiments = parse_obj(row["aspect_sentiments"])

        if aspects is None:
            continue

        if isinstance(aspects, str):
            aspects = [aspects]

        if not isinstance(aspect_sentiments, dict):
            continue

        for aspect in aspects:
            aspect = str(aspect).strip()

            if aspect == "none":
                continue

            if aspect not in ASPECTS:
                continue

            sentiment = aspect_sentiments.get(aspect)

            if sentiment not in SENTIMENTS:
                continue

            rows.append({
                "review_id": review_id,
                "review_text": review_text,
                "star_rating": row.get("star_rating", ""),
                "platform": row.get("platform", ""),
                "business_category": row.get("business_category", ""),
                "aspect": aspect,
                "sentiment": sentiment
            })

    return pd.DataFrame(rows)


df = explode_train(train_raw)

df["review_text"] = df["review_text"].apply(clean_text)
df["label"] = df["sentiment"].map(sentiment2id)

print("Aspect-level shape:", df.shape)
print(df.head())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

print("\nAspect-sentiment distribution:")
print(df.groupby(["aspect", "sentiment"]).size().sort_values())

Aspect-level shape: (3191, 8)
   review_id                                        review_text  star_rating  \
0       7238                لا يوجد الدفع بالبطاقه عند الاستلام            3   
1       7238                لا يوجد الدفع بالبطاقه عند الاستلام            3   
2       1036  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...            5   
3       1036  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...            5   
4       1036  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...            5   

      platform business_category          aspect sentiment  label  
0   play_store         ecommerce  app_experience  negative      0  
1   play_store         ecommerce        delivery  negative      0  
2  google_maps             كافيه     cleanliness  positive      2  
3  google_maps             كافيه        ambiance  positive      2  
4  google_maps             كافيه         service  positive      2  

Sentiment distribution:
sentiment
positive    1565
negative    1536
neutral     

# Build the model input shape 

In [17]:
def build_input(row):
    aspect_name = aspect_ar.get(row["aspect"], row["aspect"])

    return (
        f"التقييم: {row.get('star_rating', '')} [SEP] "
        f"المنصة: {row.get('platform', '')} [SEP] "
        f"النشاط: {row.get('business_category', '')} [SEP] "
        f"الجانب: {aspect_name} [SEP] "
        f"النص: {row['review_text']}"
    )


df["input_text"] = df.apply(build_input, axis=1)

df[["input_text", "sentiment"]].head()

,input_text,sentiment
0,التقييم: 3 [SEP] المنصة: play_store [SEP] النش...,negative
1,التقييم: 3 [SEP] المنصة: play_store [SEP] النش...,negative
2,التقييم: 5 [SEP] المنصة: google_maps [SEP] الن...,positive
3,التقييم: 5 [SEP] المنصة: google_maps [SEP] الن...,positive
4,التقييم: 5 [SEP] المنصة: google_maps [SEP] الن...,positive


# Split by rev id not by exploded 

why ? 
If you split exploded rows randomly, the same review may appear in both train and validation.

In [ ]:
import random
import pandas as pd
import re

def balance_train_dataset(
    train_df,
    group_cols=["aspect", "sentiment"],
    min_samples_per_group=60,
    max_aug_per_group=80,
    replace_prob=0.45,
    random_state=42
):
    """
    Dataset-specific synonym augmentation for your Arabic/mixed review dataset.

    It balances rare aspect-sentiment groups by creating modified copies,
    replacing frequent sentiment words found in your dataset.

    Example:
    ممتاز -> رائع
    جميل -> حلو
    سيء -> وحش
    غالي -> مبالغ فيه
    love -> adore

    Use ONLY on train_df.
    Do NOT apply to valid_df.
    """

    random.seed(random_state)

    # Built from your dataset's frequent sentiment words
    synonym_dict = {
        # =========================
        # Arabic positive - common in your data
        # =========================
        "ممتاز": ["رائع", "جميل جدا", "حلو جدا", "فوق الممتاز"],
        "ممتازة": ["رائعة", "جميلة جدا", "حلوة جدا"],
        "ممتازه": ["رائعة", "جميلة جدا", "حلوة جدا"],

        "رائع": ["ممتاز", "جميل جدا", "تحفة"],
        "رايع": ["ممتاز", "جميل جدا", "تحفة"],
        "رائعة": ["ممتازة", "جميلة جدا", "تحفة"],
        "رايعة": ["ممتازة", "جميلة جدا", "تحفة"],

        "جميل": ["حلو", "رائع", "ممتاز"],
        "جميلة": ["حلوة", "رائعة", "ممتازة"],
        "جميله": ["حلوة", "رائعة", "ممتازة"],

        "حلو": ["جميل", "كويس", "رائع"],
        "حلوه": ["جميلة", "كويسة", "رائعة"],
        "حلوة": ["جميلة", "كويسة", "رائعة"],

        "كويس": ["جيد", "حلو", "مقبول جدا"],
        "كويسة": ["جيدة", "حلوة", "مقبولة جدا"],
        "كويسه": ["جيدة", "حلوة", "مقبولة جدا"],

        "جيد": ["كويس", "مقبول", "لطيف"],
        "جيدة": ["كويسة", "مقبولة", "لطيفة"],

        "تحفه": ["رائع", "ممتاز", "جميل جدا"],
        "تحفة": ["رائع", "ممتاز", "جميل جدا"],

        "لذيذ": ["شهي", "طعمه جميل", "حلو"],
        "لذيذة": ["شهية", "طعمها جميل", "حلوة"],
        "لذيذه": ["شهية", "طعمها جميل", "حلوة"],

        "نظيف": ["مرتب", "نظيف جدا"],
        "نضيف": ["مرتب", "نظيف جدا"],
        "نظيفة": ["مرتبة", "نظيفة جدا"],
        "نضيفه": ["مرتبة", "نظيفة جدا"],

        "مناسبة": ["معقولة", "كويسة", "جيدة"],
        "مناسب": ["معقول", "كويس", "جيد"],
        "معقول": ["مناسب", "كويس"],
        "معقولة": ["مناسبة", "كويسة"],

        "محترم": ["راقي", "مهذب", "جيد"],
        "محترمة": ["راقية", "مهذبة", "جيدة"],
        "راقي": ["محترم", "ممتاز"],
        "راقية": ["محترمة", "ممتازة"],

        "سهل": ["بسيط", "واضح", "سهل الاستخدام"],
        "سهلة": ["بسيطة", "واضحة", "سهلة الاستخدام"],
        "سريع": ["سريع جدا", "في الوقت", "بدون تأخير"],
        "سريعة": ["سريعة جدا", "في الوقت", "بدون تأخير"],

        "انصح": ["أوصي", "ارشح", "أنصح"],
        "اشكر": ["أشكر", "اقدر", "أقدر"],

        # =========================
        # Arabic negative - common in your data
        # =========================
        "سيء": ["وحش", "رديء", "غير جيد"],
        "سيئ": ["وحش", "رديء", "غير جيد"],
        "سئ": ["وحش", "رديء", "غير جيد"],
        "سيية": ["وحشة", "رديئة", "غير جيدة"],
        "سييه": ["وحشة", "رديئة", "غير جيدة"],
        "سيي": ["وحش", "رديء", "غير جيد"],
        "سي": ["وحش", "رديء"],

        "وحش": ["سيء", "رديء", "غير مقبول"],
        "وحشة": ["سيئة", "رديئة", "غير مقبولة"],
        "رديء": ["سيء", "وحش", "غير جيد"],
        "رديئة": ["سيئة", "وحشة", "غير جيدة"],

        "اسوء": ["أسوأ", "سيء جدا", "رديء جدا"],
        "اسوأ": ["أسوأ", "سيء جدا", "رديء جدا"],
        "اسوا": ["أسوأ", "سيء جدا", "رديء جدا"],

        "غالي": ["مرتفع السعر", "مبالغ فيه", "سعره عالي"],
        "غالية": ["مرتفعة السعر", "مبالغ فيها", "سعرها عالي"],
        "غاليه": ["مرتفعة السعر", "مبالغ فيها", "سعرها عالي"],
        "مرتفعة": ["غالية", "مبالغ فيها"],
        "مرتفع": ["غالي", "مبالغ فيه"],
        "مبالغ": ["مرتفع", "غالي"],

        "بطيء": ["متأخر", "بطئ جدا", "فيه تأخير"],
        "بطئ": ["متأخر", "بطيء جدا", "فيه تأخير"],
        "بطيئة": ["متأخرة", "بطيئة جدا", "فيها تأخير"],
        "متأخر": ["بطيء", "متأخر جدا", "فيه تأخير"],
        "متاخر": ["بطيء", "متأخر جدا", "فيه تأخير"],

        "بارد": ["مش سخن", "غير ساخن", "وصل بارد"],
        "باردة": ["مش سخنة", "غير ساخنة", "وصلت باردة"],

        "وسخ": ["غير نظيف", "قذر", "مش نظيف"],
        "وسخة": ["غير نظيفة", "قذرة", "مش نظيفة"],
        "قذر": ["وسخ", "غير نظيف"],
        "قذرة": ["وسخة", "غير نظيفة"],

        "مزعج": ["غير مريح", "سيء", "متعب"],
        "مزعجة": ["غير مريحة", "سيئة", "متعبة"],

        "مفيش": ["لا يوجد", "مش موجود"],
        "يوجد": ["متوفر", "موجود"],

        # app/delivery phrases from your data
        "يعمل": ["يشتغل"],
        "شغال": ["يعمل"],
        "الدفع": ["عملية الدفع"],
        "التحديث": ["اخر تحديث"],
        "التوصيل": ["الدليفري", "توصيل الطلب"],
        "اوردر": ["طلب", "أوردر"],
        "الاوردر": ["الطلب", "الأوردر"],

        # =========================
        # Neutral - common in your data
        # =========================
        "عادي": ["مقبول", "متوسط", "لا بأس به"],
        "عادية": ["مقبولة", "متوسطة", "لا بأس بها"],
        "مقبول": ["عادي", "متوسط", "لا بأس به"],
        "مقبولة": ["عادية", "متوسطة", "لا بأس بها"],
        "متوسط": ["عادي", "مقبول"],
        "متوسطة": ["عادية", "مقبولة"],
        "تمام": ["مقبول", "لا بأس به"],

        # =========================
        # English / mixed reviews found in your data
        # =========================
        "good": ["great", "nice", "decent"],
        "great": ["excellent", "amazing", "wonderful"],
        "excellent": ["amazing", "outstanding", "great"],
        "amazing": ["excellent", "wonderful", "great"],
        "nice": ["good", "pleasant", "lovely"],
        "perfect": ["excellent", "amazing", "great"],
        "love": ["adore", "really like", "enjoy"],
        "loved": ["adored", "really liked", "enjoyed"],

        "bad": ["poor", "terrible", "awful"],
        "terrible": ["awful", "very bad", "horrible"],
        "awful": ["terrible", "horrible", "very bad"],
        "slow": ["delayed", "very slow", "late"],
        "expensive": ["overpriced", "costly", "too expensive"],
        "dirty": ["unclean", "not clean", "filthy"],

        "ok": ["average", "acceptable", "fine"],
        "okay": ["average", "acceptable", "fine"],
        "average": ["acceptable", "okay", "normal"],
    }

    # Do not touch these. They can flip sentiment.
    protected_words = {
        "مش", "مو", "لا", "ما", "لم", "لن", "ليس", "ليست", "بدون", "غير",
        "not", "no", "never", "n't",
        "but", "لكن", "بس", "ولكن"
    }

    # Avoid replacing aspect nouns too aggressively.
    # We want to replace sentiment words, not the aspect itself.
    protected_aspect_words = {
        "الاكل", "اكل", "الطعام", "طعام", "الطعم", "طعم",
        "الخدمة", "خدمة", "الخدمه", "خدمه",
        "السعر", "سعر", "الاسعار", "اسعار",
        "النظافة", "نظافة", "النضافه",
        "التوصيل", "توصيل", "الدليفري",
        "المكان", "مكان", "الجو", "جو",
        "التطبيق", "تطبيق", "برنامج", "البرنامج"
    }

    def replace_phrases(text):
        """
        Dataset-specific phrase replacements.
        These are safer for common phrases.
        """
        phrase_replacements = {
            "فوق الممتاز": ["ممتاز جدا", "رائع جدا"],
            "مبالغ فيه": ["غالي جدا", "مرتفع السعر"],
            "مبالغ فيها": ["غالية جدا", "مرتفعة السعر"],
            "لا يعمل": ["مش شغال", "لا يشتغل"],
            "مش شغال": ["لا يعمل", "لا يشتغل"],
            "غير نظيف": ["مش نظيف", "وسخ"],
            "غير نظيفة": ["مش نظيفة", "وسخة"],
            "لا انصح": ["مش برشح", "لا أوصي"],
            "مش انصح": ["لا أوصي", "مش برشح"],
            "لا باس به": ["عادي", "مقبول"],
            "لا بأس به": ["عادي", "مقبول"],
            "لا بأس بها": ["عادية", "مقبولة"],
        }

        changed = False

        for phrase, replacements in phrase_replacements.items():
            if phrase in text and random.random() < replace_prob:
                text = text.replace(phrase, random.choice(replacements), 1)
                changed = True

        return text, changed

    def synonym_replace_text(text):
        text = str(text)

        # Try phrase replacement first
        text_after_phrase, phrase_changed = replace_phrases(text)

        words = text_after_phrase.split()
        new_words = []
        word_changed = False

        for word in words:
            clean_word = word.strip(".,!?؟،؛:()[]{}\"'").lower()

            if clean_word in protected_words:
                new_words.append(word)
                continue

            if clean_word in protected_aspect_words:
                new_words.append(word)
                continue

            replacement = None

            # Exact Arabic or lowercase English
            if clean_word in synonym_dict and random.random() < replace_prob:
                replacement = random.choice(synonym_dict[clean_word])

            if replacement is not None:
                new_words.append(replacement)
                word_changed = True
            else:
                new_words.append(word)

        final_text = " ".join(new_words)

        changed = phrase_changed or word_changed
        return final_text, changed

    df = train_df.copy()
    df["is_augmented"] = 0

    augmented_rows = []

    group_sizes = df.groupby(group_cols).size().sort_values()
    print("Before augmentation:")
    print(group_sizes)

    for group_key, group in df.groupby(group_cols):
        n = len(group)

        if n >= min_samples_per_group:
            continue

        needed = min(min_samples_per_group - n, max_aug_per_group)

        sampled = group.sample(
            n=needed,
            replace=True,
            random_state=random_state
        )

        for _, row in sampled.iterrows():
            original_text = str(row["review_text"])

            changed = False
            new_text = original_text

            # Try more than once to avoid no-change augmentations
            for _ in range(8):
                candidate_text, changed = synonym_replace_text(original_text)

                if changed and candidate_text != original_text:
                    new_text = candidate_text
                    break

            if not changed or new_text == original_text:
                continue

            new_row = row.copy()
            new_row["review_text"] = new_text
            new_row["is_augmented"] = 1

            # Rebuild input_text if build_input exists
            if "input_text" in train_df.columns:
                try:
                    new_row["input_text"] = build_input(new_row)
                except:
                    pass

            augmented_rows.append(new_row)

    if len(augmented_rows) > 0:
        aug_df = pd.DataFrame(augmented_rows)
        balanced_df = pd.concat([df, aug_df], ignore_index=True)
    else:
        balanced_df = df.copy()

    balanced_df = balanced_df.sample(
        frac=1,
        random_state=random_state
    ).reset_index(drop=True)

    print("\nAugmented rows added:", len(augmented_rows))

    print("\nAfter augmentation:")
    print(balanced_df.groupby(group_cols).size().sort_values())

    print("\nOld shape:", train_df.shape)
    print("New shape:", balanced_df.shape)

    return balanced_df

In [19]:
train_raw = train.drop_duplicates(
    subset=["review_text", "aspects", "aspect_sentiments"]
).reset_index(drop=True)

valid_raw = validation.copy()

train_df = explode_train(train_raw)
valid_df = explode_train(valid_raw)

# Clean + labels + input for both train and validation
for temp_df in [train_df, valid_df]:
    temp_df["review_text"] = temp_df["review_text"].apply(clean_text)
    temp_df["label"] = temp_df["sentiment"].map(sentiment2id)
    temp_df["input_text"] = temp_df.apply(build_input, axis=1)

print("Before augmentation:")
print("Train aspect rows:", train_df.shape)
print("Valid aspect rows:", valid_df.shape)

print("\nOriginal train sentiment distribution:")
print(train_df["sentiment"].value_counts())

print("\nOriginal train aspect-sentiment distribution:")
print(train_df.groupby(["aspect", "sentiment"]).size().sort_values())

# Balance / augment ONLY train_df
# The output is assigned back to train_df so the rest of your notebook stays the same
train_df = balance_train_dataset(
    train_df,
    group_cols=["aspect", "sentiment"],
    min_samples_per_group=40,   # start safe
    max_aug_per_group=60,
    replace_prob=0.35,
    random_state=SEED
)

# Make sure labels and input_text are still correct after augmentation
train_df["review_text"] = train_df["review_text"].apply(clean_text)
train_df["label"] = train_df["sentiment"].map(sentiment2id)
train_df["input_text"] = train_df.apply(build_input, axis=1)

# Do not balance validation
valid_df["label"] = valid_df["sentiment"].map(sentiment2id)
valid_df["input_text"] = valid_df.apply(build_input, axis=1)

print("\nAfter augmentation:")
print("Train aspect rows:", train_df.shape)
print("Valid aspect rows:", valid_df.shape)

print("\nTrain sentiment distribution:")
print(train_df["sentiment"].value_counts())

print("\nValid sentiment distribution:")
print(valid_df["sentiment"].value_counts())

print("\nTrain aspect-sentiment distribution:")
print(train_df.groupby(["aspect", "sentiment"]).size().sort_values())

print("\nValid aspect-sentiment distribution:")
print(valid_df.groupby(["aspect", "sentiment"]).size().sort_values())

print("\nAugmented rows count:")
if "is_augmented" in train_df.columns:
    print(train_df["is_augmented"].value_counts())
else:
    print("No is_augmented column found.")

Before augmentation:
Train aspect rows: (3191, 9)
Valid aspect rows: (828, 9)

Original train sentiment distribution:
sentiment
positive    1565
negative    1536
neutral       90
Name: count, dtype: int64

Original train aspect-sentiment distribution:
aspect          sentiment
cleanliness     neutral        1
delivery        neutral        1
ambiance        neutral        9
service         neutral       10
price           neutral       11
general         neutral       12
app_experience  neutral       15
delivery        positive      18
food            neutral       31
general         negative      32
cleanliness     negative      75
ambiance        negative     101
app_experience  positive     109
cleanliness     positive     109
price           positive     110
delivery        negative     142
general         positive     176
food            negative     177
price           negative     233
food            positive     246
ambiance        positive     268
app_experience  negative     

# Prepare text for Masked LLM

In [20]:
mlm_texts = pd.concat([
    train_raw["review_text"],
    unlabeled["review_text"]
], ignore_index=True)

mlm_texts = (
    mlm_texts
    .dropna()
    .astype(str)
    .apply(clean_text)
)

mlm_texts = mlm_texts[mlm_texts.str.len() > 2]
mlm_texts = mlm_texts.drop_duplicates().reset_index(drop=True)

print("MLM texts:", len(mlm_texts))
mlm_texts.head()

MLM texts: 7672


0                  لا يوجد الدفع بالبطاقه عند الاستلام
1    المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...
2    تجربة سيية سالتهم الاكل هياخد وقت قد ايه قالول...
3                                      احلي مكان فزايد
4    الفطير حلو جدا الاحجام تحفة بالنسبه للسعر فا ي...
Name: review_text, dtype: object

# Define models we will be using 

In [21]:
BASE_MLM_MODEL = "xlm-roberta-base"
DOMAIN_MODEL_DIR = "./domain_xlm_roberta_reviews"

## Tokenize data for model 

In [22]:
mlm_tokenizer = AutoTokenizer.from_pretrained(BASE_MLM_MODEL)

mlm_dataset = Dataset.from_dict({
    "text": mlm_texts.tolist()
})

def tokenize_for_mlm(batch):
    return mlm_tokenizer(
        batch["text"],
        truncation=True,
        max_length=160
    )

mlm_dataset = mlm_dataset.map(
    tokenize_for_mlm,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/7672 [00:00<?, ? examples/s]

## init the data collator 

In [23]:
mlm_collator = DataCollatorForLanguageModeling(
    tokenizer=mlm_tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# Train the model 

In [24]:
# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("No GPU detected. Training will use CPU (much slower).")

CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [26]:
device = torch.device("cuda")

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [28]:
mlm_model = AutoModelForMaskedLM.from_pretrained(BASE_MLM_MODEL)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mlm_model = mlm_model.to(device)
print(f"Model moved to: {device}")

mlm_args = TrainingArguments(
    output_dir=DOMAIN_MODEL_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    gradient_checkpointing=True,
    use_cpu=False
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=mlm_dataset,
    data_collator=mlm_collator
)

mlm_trainer.train()

mlm_trainer.save_model(DOMAIN_MODEL_DIR)
mlm_tokenizer.save_pretrained(DOMAIN_MODEL_DIR)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model moved to: cuda


Step,Training Loss
100,18.070983
200,16.260308
300,14.977620
400,16.064128
500,14.623746
600,14.818856
700,14.140603
800,15.488962
900,14.864812
1000,14.088794


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./domain_xlm_roberta_reviews/tokenizer_config.json',
 './domain_xlm_roberta_reviews/tokenizer.json')

# Use weighted class approach for non balanced data 


In [29]:
# Use class weights because of the non balance bet data 
def get_class_weights(labels):
    classes = np.array([0, 1, 2])

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=labels
    )

    return torch.tensor(weights, dtype=torch.float)

In [30]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = self.class_weights.to(logits.device)

        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, 3), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [31]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds)
    }

# Create the sentiment transformer 

In [32]:
def train_transformer_model(
    model_name,
    train_df,
    valid_df,
    output_dir,
    epochs=4,
    lr=2e-5,
    batch_size=8,
    max_length=160
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_ds = Dataset.from_pandas(
        train_df[["input_text", "label"]].reset_index(drop=True)
    )

    valid_ds = Dataset.from_pandas(
        valid_df[["input_text", "label"]].reset_index(drop=True)
    )

    def tokenize_batch(batch):
        return tokenizer(
            batch["input_text"],
            truncation=True,
            max_length=max_length
        )

    train_ds = train_ds.map(tokenize_batch, batched=True)
    valid_ds = valid_ds.map(tokenize_batch, batched=True)

    train_ds = train_ds.rename_column("label", "labels")
    valid_ds = valid_ds.rename_column("label", "labels")

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2sentiment,
        label2id=sentiment2id
    )

    class_weights = get_class_weights(train_df["label"].values)

    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        save_total_limit=1,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED
    )

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    return trainer, tokenizer

# Prop prediction function 

In [33]:
def predict_proba_transformer(trainer, tokenizer, texts, max_length=160):
    ds = Dataset.from_dict({
        "input_text": list(texts)
    })

    def tokenize_batch(batch):
        return tokenizer(
            batch["input_text"],
            truncation=True,
            max_length=max_length
        )

    ds = ds.map(tokenize_batch, batched=True)

    preds = trainer.predict(ds)
    logits = preds.predictions

    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)

    return probs

# Train domain model xlim roberta 

In [35]:
trainer_xlm, tokenizer_xlm = train_transformer_model(
    model_name=DOMAIN_MODEL_DIR,
    train_df=train_df,
    valid_df=valid_df,
    output_dir="./sentiment_domain_xlm",
    epochs=7,
    lr=2e-5,
    batch_size=8,
    max_length=160
)

valid_probs_xlm = predict_proba_transformer(
    trainer_xlm,
    tokenizer_xlm,
    valid_df["input_text"]
)

valid_pred_xlm = valid_probs_xlm.argmax(axis=1)

print("Domain XLM-R Macro F1:", f1_score(valid_df["label"], valid_pred_xlm, average="macro"))
print(classification_report(valid_df["label"], valid_pred_xlm, target_names=SENTIMENTS))

Map:   0%|          | 0/3385 [00:00<?, ? examples/s]

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: ./domain_xlm_roberta_reviews
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.731352,0.798655,0.659058,0.897343
2,0.465144,0.613773,0.647764,0.856280
3,0.432109,0.654771,0.653235,0.863527
4,0.412364,0.831066,0.682520,0.911836
5,0.508055,0.855344,0.669681,0.904589
6,0.283596,0.856957,0.644363,0.888889
7,0.364905,0.827282,0.650840,0.885266


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

Domain XLM-R Macro F1: 0.6825199400777183
              precision    recall  f1-score   support

    negative       0.96      0.90      0.93       357
     neutral       0.27      0.14      0.19        28
    positive       0.90      0.97      0.93       443

    accuracy                           0.91       828
   macro avg       0.71      0.67      0.68       828
weighted avg       0.90      0.91      0.91       828



In [36]:
valid_probs_xlm = predict_proba_transformer(
    trainer_xlm,
    tokenizer_xlm,
    valid_df["input_text"]
)

valid_pred_xlm = valid_probs_xlm.argmax(axis=1)

print("Domain XLM-R Macro F1:", f1_score(valid_df["label"], valid_pred_xlm, average="macro"))
print(classification_report(valid_df["label"], valid_pred_xlm, target_names=SENTIMENTS))

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

Domain XLM-R Macro F1: 0.6825199400777183
              precision    recall  f1-score   support

    negative       0.96      0.90      0.93       357
     neutral       0.27      0.14      0.19        28
    positive       0.90      0.97      0.93       443

    accuracy                           0.91       828
   macro avg       0.71      0.67      0.68       828
weighted avg       0.90      0.91      0.91       828



# Third model

In [37]:
MARBERT_MODEL = "UBC-NLP/MARBERTv2"

In [38]:
trainer_marbert, tokenizer_marbert = train_transformer_model(
    model_name=MARBERT_MODEL,
    train_df=train_df,
    valid_df=valid_df,
    output_dir="./sentiment_marbert",
    epochs=6,
    lr=2e-5,
    batch_size=8,
    max_length=160
)

Map:   0%|          | 0/3385 [00:00<?, ? examples/s]

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were ne

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.480809,0.633463,0.696253,0.910628
2,0.308888,0.527024,0.696179,0.891304
3,0.203584,0.724914,0.710013,0.915459
4,0.186589,0.749742,0.725716,0.915459
5,0.115176,0.785220,0.765870,0.932367
6,0.019597,0.752895,0.770030,0.931159


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [39]:
valid_probs_marbert = predict_proba_transformer(
    trainer_marbert,
    tokenizer_marbert,
    valid_df["input_text"]
)

valid_pred_marbert = valid_probs_marbert.argmax(axis=1)

print("MARBERT Macro F1:", f1_score(valid_df["label"], valid_pred_marbert, average="macro"))
print(classification_report(valid_df["label"], valid_pred_marbert, target_names=SENTIMENTS))

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

MARBERT Macro F1: 0.7700299542984395
              precision    recall  f1-score   support

    negative       0.94      0.94      0.94       357
     neutral       0.44      0.39      0.42        28
    positive       0.95      0.96      0.95       443

    accuracy                           0.93       828
   macro avg       0.78      0.76      0.77       828
weighted avg       0.93      0.93      0.93       828



# Fourth model 

In [40]:
CAMEL_MODEL = "CAMeL-Lab/bert-base-arabic-camelbert-mix"

In [42]:
trainer_camel, tokenizer_camel = train_transformer_model(
    model_name=CAMEL_MODEL,
    train_df=train_df,
    valid_df=valid_df,
    output_dir="./sentiment_camelbert",
    epochs=6,
    batch_size=8,
    max_length=160
)

Map:   0%|          | 0/3385 [00:00<?, ? examples/s]

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSIN

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.460153,0.580350,0.649026,0.902174
2,0.326376,0.561937,0.688708,0.894928
3,0.221432,0.647447,0.701066,0.905797
4,0.193586,0.821958,0.682175,0.920290
5,0.199271,0.823696,0.711445,0.917874
6,0.133025,0.817159,0.688767,0.914251


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [43]:
valid_probs_camel = predict_proba_transformer(
    trainer_camel,
    tokenizer_camel,
    valid_df["input_text"]
)

valid_pred_camel = valid_probs_camel.argmax(axis=1)

print("CAMeLBERT Macro F1:", f1_score(valid_df["label"], valid_pred_camel, average="macro"))
print(classification_report(valid_df["label"], valid_pred_camel, target_names=SENTIMENTS))

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

CAMeLBERT Macro F1: 0.711444852362794
              precision    recall  f1-score   support

    negative       0.92      0.93      0.93       357
     neutral       0.56      0.18      0.27        28
    positive       0.92      0.95      0.94       443

    accuracy                           0.92       828
   macro avg       0.80      0.69      0.71       828
weighted avg       0.91      0.92      0.91       828



In [44]:
f1_xlm = f1_score(valid_df["label"], valid_pred_xlm, average="macro")
f1_marbert = f1_score(valid_df["label"], valid_pred_marbert, average="macro")
f1_camel = f1_score(valid_df["label"], valid_pred_camel, average="macro")

scores = np.array([f1_xlm, f1_marbert, f1_camel])
weights = scores / scores.sum()

print("Scores:", scores)
print("Weights:", weights)

valid_probs_ensemble = (
    weights[0] * valid_probs_xlm +
    weights[1] * valid_probs_marbert +
    weights[2] * valid_probs_camel
)

valid_pred_ensemble = valid_probs_ensemble.argmax(axis=1)

print("Weighted Ensemble Macro F1:", f1_score(valid_df["label"], valid_pred_ensemble, average="macro"))
print(classification_report(valid_df["label"], valid_pred_ensemble, target_names=SENTIMENTS))

Scores: [0.68251994 0.77002995 0.71144485]
Weights: [0.31539815 0.35583726 0.32876459]
Weighted Ensemble Macro F1: 0.7372784163625813
              precision    recall  f1-score   support

    negative       0.95      0.93      0.94       357
     neutral       0.47      0.25      0.33        28
    positive       0.93      0.97      0.95       443

    accuracy                           0.93       828
   macro avg       0.78      0.72      0.74       828
weighted avg       0.92      0.93      0.92       828



In [45]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd
import json
import ast

# XLM validation already exists in your code, but safe to recompute
valid_probs_xlm = predict_proba_transformer(
    trainer_xlm,
    tokenizer_xlm,
    valid_df["input_text"]
)

valid_probs_marbert = predict_proba_transformer(
    trainer_marbert,
    tokenizer_marbert,
    valid_df["input_text"]
)

valid_probs_camel = predict_proba_transformer(
    trainer_camel,
    tokenizer_camel,
    valid_df["input_text"]
)

valid_pred_xlm = valid_probs_xlm.argmax(axis=1)
valid_pred_marbert = valid_probs_marbert.argmax(axis=1)
valid_pred_camel = valid_probs_camel.argmax(axis=1)

f1_xlm = f1_score(valid_df["label"], valid_pred_xlm, average="macro")
f1_marbert = f1_score(valid_df["label"], valid_pred_marbert, average="macro")
f1_camel = f1_score(valid_df["label"], valid_pred_camel, average="macro")

print("Domain XLM-R Macro F1:", f1_xlm)
print("MARBERT Macro F1:", f1_marbert)
print("CAMeLBERT Macro F1:", f1_camel)

print("\nDomain XLM-R report:")
print(classification_report(valid_df["label"], valid_pred_xlm, target_names=SENTIMENTS))

print("\nMARBERT report:")
print(classification_report(valid_df["label"], valid_pred_marbert, target_names=SENTIMENTS))

print("\nCAMeLBERT report:")
print(classification_report(valid_df["label"], valid_pred_camel, target_names=SENTIMENTS))

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

Domain XLM-R Macro F1: 0.6825199400777183
MARBERT Macro F1: 0.7700299542984395
CAMeLBERT Macro F1: 0.711444852362794

Domain XLM-R report:
              precision    recall  f1-score   support

    negative       0.96      0.90      0.93       357
     neutral       0.27      0.14      0.19        28
    positive       0.90      0.97      0.93       443

    accuracy                           0.91       828
   macro avg       0.71      0.67      0.68       828
weighted avg       0.90      0.91      0.91       828


MARBERT report:
              precision    recall  f1-score   support

    negative       0.94      0.94      0.94       357
     neutral       0.44      0.39      0.42        28
    positive       0.95      0.96      0.95       443

    accuracy                           0.93       828
   macro avg       0.78      0.76      0.77       828
weighted avg       0.93      0.93      0.93       828


CAMeLBERT report:
              precision    recall  f1-score   support

    nega

In [46]:
weight_grid = [
    (0.40, 0.40, 0.20),
    (0.45, 0.40, 0.15),
    (0.50, 0.35, 0.15),
    (0.35, 0.45, 0.20),
    (0.33, 0.33, 0.34),
    (0.50, 0.25, 0.25),
    (0.25, 0.50, 0.25),
    (0.25, 0.25, 0.50),
]

best_f1 = -1
best_weights = None
best_valid_probs = None
best_valid_pred = None

for w in weight_grid:
    probs = (
        w[0] * valid_probs_xlm +
        w[1] * valid_probs_marbert +
        w[2] * valid_probs_camel
    )
    
    pred = probs.argmax(axis=1)
    score = f1_score(valid_df["label"], pred, average="macro")
    
    print("Weights:", w, "Macro F1:", score)
    
    if score > best_f1:
        best_f1 = score
        best_weights = w
        best_valid_probs = probs
        best_valid_pred = pred

print("\nBest ensemble weights:", best_weights)
print("Best ensemble Macro F1:", best_f1)

print("\nBest ensemble report:")
print(classification_report(valid_df["label"], best_valid_pred, target_names=SENTIMENTS))

Weights: (0.4, 0.4, 0.2) Macro F1: 0.7428675080803595
Weights: (0.45, 0.4, 0.15) Macro F1: 0.742021177086841
Weights: (0.5, 0.35, 0.15) Macro F1: 0.7279924826370078
Weights: (0.35, 0.45, 0.2) Macro F1: 0.7480615983966398
Weights: (0.33, 0.33, 0.34) Macro F1: 0.7372784163625813
Weights: (0.5, 0.25, 0.25) Macro F1: 0.7142619232204899
Weights: (0.25, 0.5, 0.25) Macro F1: 0.7729813347735064
Weights: (0.25, 0.25, 0.5) Macro F1: 0.7290020083213142

Best ensemble weights: (0.25, 0.5, 0.25)
Best ensemble Macro F1: 0.7729813347735064

Best ensemble report:
              precision    recall  f1-score   support

    negative       0.94      0.96      0.95       357
     neutral       0.48      0.36      0.41        28
    positive       0.96      0.96      0.96       443

    accuracy                           0.94       828
   macro avg       0.79      0.76      0.77       828
weighted avg       0.94      0.94      0.94       828



In [47]:
model_scores = {
    "domain_xlm": f1_xlm,
    "marbert": f1_marbert,
    "camel": f1_camel,
    "ensemble": best_f1
}

print(model_scores)

BEST_MODE = max(model_scores, key=model_scores.get)
print("BEST_MODE:", BEST_MODE)

{'domain_xlm': 0.6825199400777183, 'marbert': 0.7700299542984395, 'camel': 0.711444852362794, 'ensemble': 0.7729813347735064}
BEST_MODE: ensemble


# Testing

In [55]:
TEST_PATH = "DeepX_hidden_test.xlsx"   

test = pd.read_excel(TEST_PATH)

print(test.shape)
print(test.columns)
test.head()

(1971, 8)
Index(['review_id', 'review_text', 'star_rating', 'date', 'business_name',
       'business_category', 'platform', 'aspects'],
      dtype='object')


,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]"
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]"
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,1,قبل شهر,بيت لحم Beet Lahm,مطعم,google_maps,"[""service"", ""delivery"", ""food""]"
3,3024,احلي مكان فزايد,5,قبل شهر,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,"[""general""]"
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,4,قبل سنة,The Best Restaurant,مطعم,google_maps,"[""food"", ""price""]"


In [56]:
def explode_test(df):
    rows = []

    for _, row in df.iterrows():
        review_id = row["review_id"]
        review_text = clean_text(row["review_text"])

        aspects = parse_obj(row["aspects"])

        if aspects is None:
            aspects = []

        if isinstance(aspects, str):
            aspects = [aspects]

        for aspect in aspects:
            aspect = str(aspect).strip()

            if aspect == "none":
                continue

            if aspect not in ASPECTS:
                continue

            rows.append({
                "review_id": review_id,
                "review_text": review_text,
                "star_rating": row.get("star_rating", ""),
                "platform": row.get("platform", ""),
                "business_category": row.get("business_category", ""),
                "aspect": aspect
            })

    return pd.DataFrame(rows)


test_df = explode_test(test)
test_df["input_text"] = test_df.apply(build_input, axis=1)

print(test_df.shape)
test_df.head()

(3276, 7)


,review_id,review_text,star_rating,platform,business_category,aspect,input_text
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,play_store,ecommerce,app_experience,التقييم: 3 [SEP] المنصة: play_store [SEP] النش...
1,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,play_store,ecommerce,delivery,التقييم: 3 [SEP] المنصة: play_store [SEP] النش...
2,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,google_maps,كافيه,cleanliness,التقييم: 5 [SEP] المنصة: google_maps [SEP] الن...
3,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,google_maps,كافيه,ambiance,التقييم: 5 [SEP] المنصة: google_maps [SEP] الن...
4,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,google_maps,كافيه,service,التقييم: 5 [SEP] المنصة: google_maps [SEP] الن...


In [57]:
test_probs_xlm = predict_proba_transformer(
    trainer_xlm,
    tokenizer_xlm,
    test_df["input_text"]
)

test_probs_marbert = predict_proba_transformer(
    trainer_marbert,
    tokenizer_marbert,
    test_df["input_text"]
)

test_probs_camel = predict_proba_transformer(
    trainer_camel,
    tokenizer_camel,
    test_df["input_text"]
)

Map:   0%|          | 0/3276 [00:00<?, ? examples/s]

Map:   0%|          | 0/3276 [00:00<?, ? examples/s]

Map:   0%|          | 0/3276 [00:00<?, ? examples/s]

In [58]:
if BEST_MODE == "domain_xlm":
    final_test_probs = test_probs_xlm

elif BEST_MODE == "marbert":
    final_test_probs = test_probs_marbert

elif BEST_MODE == "camel":
    final_test_probs = test_probs_camel

elif BEST_MODE == "ensemble":
    final_test_probs = (
        best_weights[0] * test_probs_xlm +
        best_weights[1] * test_probs_marbert +
        best_weights[2] * test_probs_camel
    )

elif BEST_MODE == "ensemble_boosted":
    final_test_probs = (
        best_weights[0] * test_probs_xlm +
        best_weights[1] * test_probs_marbert +
        best_weights[2] * test_probs_camel
    )
    final_test_probs = apply_neutral_boost(final_test_probs, best_boost)

else:
    raise ValueError("Unknown BEST_MODE")

test_pred_ids = final_test_probs.argmax(axis=1)
test_df["predicted_sentiment"] = [id2sentiment[i] for i in test_pred_ids]

print(test_df["predicted_sentiment"].value_counts())
test_df.head()

predicted_sentiment
positive    1656
negative    1523
neutral       97
Name: count, dtype: int64


,review_id,review_text,star_rating,platform,business_category,aspect,input_text,predicted_sentiment
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,play_store,ecommerce,app_experience,التقييم: 3 [SEP] المنصة: play_store [SEP] النش...,negative
1,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,play_store,ecommerce,delivery,التقييم: 3 [SEP] المنصة: play_store [SEP] النش...,negative
2,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,google_maps,كافيه,cleanliness,التقييم: 5 [SEP] المنصة: google_maps [SEP] الن...,positive
3,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,google_maps,كافيه,ambiance,التقييم: 5 [SEP] المنصة: google_maps [SEP] الن...,positive
4,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,google_maps,كافيه,service,التقييم: 5 [SEP] المنصة: google_maps [SEP] الن...,positive


In [59]:
def build_submission(test_raw, test_aspect_df):
    pred_map = {}

    for _, row in test_aspect_df.iterrows():
        review_id = int(row["review_id"])
        aspect = str(row["aspect"]).strip()
        sentiment = str(row["predicted_sentiment"]).strip()

        if aspect == "none":
            continue

        if aspect not in ASPECTS:
            continue

        if sentiment not in SENTIMENTS:
            continue

        if review_id not in pred_map:
            pred_map[review_id] = {}

        pred_map[review_id][aspect] = sentiment

    submission = []

    for _, row in test_raw.iterrows():
        review_id = int(row["review_id"])
        aspects = parse_obj(row["aspects"])

        if aspects is None:
            aspects = []

        if isinstance(aspects, str):
            aspects = [aspects]

        clean_aspects = []
        aspect_sentiments = {}

        for aspect in aspects:
            aspect = str(aspect).strip()

            if aspect == "none":
                continue

            if aspect not in ASPECTS:
                continue

            if review_id in pred_map and aspect in pred_map[review_id]:
                clean_aspects.append(aspect)
                aspect_sentiments[aspect] = pred_map[review_id][aspect]

        submission.append({
            "review_id": review_id,
            "aspects": clean_aspects,
            "aspect_sentiments": aspect_sentiments
        })

    return submission


submission = build_submission(test, test_df)

print(submission[:3])

[{'review_id': 7238, 'aspects': ['app_experience', 'delivery'], 'aspect_sentiments': {'app_experience': 'negative', 'delivery': 'negative'}}, {'review_id': 1036, 'aspects': ['cleanliness', 'ambiance', 'service'], 'aspect_sentiments': {'cleanliness': 'positive', 'ambiance': 'positive', 'service': 'positive'}}, {'review_id': 1975, 'aspects': ['service', 'delivery', 'food'], 'aspect_sentiments': {'service': 'negative', 'delivery': 'negative', 'food': 'neutral'}}]


In [60]:
def validate_submission(submission):
    allowed_aspects = [
        "food",
        "service",
        "price",
        "cleanliness",
        "delivery",
        "ambiance",
        "app_experience",
        "general"
    ]

    allowed_sentiments = ["positive", "negative", "neutral"]

    assert isinstance(submission, list), "Submission must be a list"

    seen_ids = set()

    for i, item in enumerate(submission):
        assert isinstance(item, dict), f"Item {i} must be a dictionary"

        assert "review_id" in item, f"Missing review_id at item {i}"
        assert "aspects" in item, f"Missing aspects at item {i}"
        assert "aspect_sentiments" in item, f"Missing aspect_sentiments at item {i}"

        assert isinstance(item["review_id"], int), f"review_id must be int at item {i}"
        assert isinstance(item["aspects"], list), f"aspects must be list at item {i}"
        assert isinstance(item["aspect_sentiments"], dict), f"aspect_sentiments must be dict at item {i}"

        assert item["review_id"] not in seen_ids, f"Duplicate review_id {item['review_id']}"
        seen_ids.add(item["review_id"])

        assert len(item["aspects"]) == len(set(item["aspects"])), f"Duplicate aspects at item {i}"

        for aspect in item["aspects"]:
            assert aspect in allowed_aspects, f"Invalid aspect '{aspect}' at item {i}"
            assert aspect in item["aspect_sentiments"], f"Missing sentiment for aspect '{aspect}' at item {i}"
            assert item["aspect_sentiments"][aspect] in allowed_sentiments, f"Invalid sentiment for aspect '{aspect}' at item {i}"

        for aspect in item["aspect_sentiments"]:
            assert aspect in item["aspects"], f"Aspect '{aspect}' in aspect_sentiments but not in aspects at item {i}"

    print("Submission is valid.")
    print("Rows:", len(submission))


validate_submission(submission)

Submission is valid.
Rows: 1971


In [61]:
with open("submission.json", "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print("Saved submission.json")

Saved submission.json
